# No ROS

In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import os

# Defining the path for the csv file
path = os.path.join("dataset.csv")

# Storing the dataframe in a variable named dataset
dataset = pd.read_csv(path)

# Dropping the unnecessary columns
dataset = dataset.drop('seqn', axis='columns')
dataset = dataset.drop('Marital', axis='columns')

In [ ]:
# Encoding the categorical variables and filling in the missing values
sex_mapping = {'Male': 0, 'Female': 1}
race_mapping = {'White': 0, 'Asian': 1, 'Black': 2, 'MexAmerican': 3, 'Hispanic': 4, 'Other': 5}

dataset['Sex'] = dataset['Sex'].replace(sex_mapping)
dataset['Race'] = dataset['Race'].replace(race_mapping)

# Fill NaN values in column with index 2, 4, and 5
dataset.iloc[:, 2] = dataset.iloc[:, 2].fillna(dataset.iloc[:, 2].mean())
dataset.iloc[:, 4] = dataset.iloc[:, 4].fillna(dataset.iloc[:, 4].mean())
dataset.iloc[:, 5] = dataset.iloc[:, 5].fillna(dataset.iloc[:, 5].mean())

In [ ]:
# Define the features and target variable
x = dataset.drop('MetabolicSyndrome', axis=1)
y = dataset['MetabolicSyndrome']

# Perform random over sampling

# Combine the resampled features and target variable into a new dataframe
# Splitting into training and testing data
outcome_0 = dataset[dataset['MetabolicSyndrome'] == 0]
outcome_1 = dataset[dataset['MetabolicSyndrome'] == 1]


test_size_each_class = 474
test_0 = outcome_0.sample(n=test_size_each_class, random_state=42)
test_1 = outcome_1.sample(n=test_size_each_class, random_state=42)
test_data = pd.concat([test_0, test_1])

# Remove the test set rows from the original dataset to create the training set
train_data = dataset.drop(test_data.index)


x_train = train_data.drop('MetabolicSyndrome', axis=1).values
y_train = train_data['MetabolicSyndrome'].values
x_test = test_data.drop('MetabolicSyndrome', axis=1).values
y_test = test_data['MetabolicSyndrome'].values


In [ ]:
random_forest_classifier = RandomForestClassifier(n_estimators=30, criterion='entropy', random_state=0)
classifier_decision_tree = DecisionTreeClassifier(criterion='entropy')
xgb_classifer = XGBClassifier(n_estimators = 100, max_depth = 3, learning_rate = 0.5)
classifier_logistic_regression = LogisticRegression()
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=10)
tabnet = TabNetClassifier()

random_forest_classifier.fit(x_train, y_train)
classifier_decision_tree.fit(x_train, y_train)
xgb_classifer.fit(x_train, y_train)
classifier_logistic_regression.fit(x_train, y_train)
mlp.fit(x_train, y_train)
tabnet.fit(
    x_train, y_train,
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

tabnet_pred = tabnet.predict(x_test)
rf_pred = random_forest_classifier.predict(x_test)
dt_pred = classifier_decision_tree.predict(x_test)
mlp_pred = mlp.predict(x_test)
tabnet_pred = tabnet.predict(x_test)
xgb_pred = xgb_classifer.predict(x_test)

# Function to evaluate the model
def evaluate_model(model, x_test, y_test):
    accuracy = []
    precision = []
    recall = []
    f1 = []
    for _ in range(3):
        y_pred = model.predict(x_test)
        accuracy.append(accuracy_score(y_test, y_pred))
        precision.append(precision_score(y_test, y_pred))
        recall.append(recall_score(y_test, y_pred))
        f1.append(f1_score(y_test, y_pred))
    return np.mean(accuracy), np.mean(precision), np.mean(recall), np.mean(f1)

# Evaluate each model
results = {
    'Model': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1 Score': []
}

models = {
    'Random Forest': random_forest_classifier,
    'Decision Tree': classifier_decision_tree,
    'XGBoost': xgb_classifer,
    'Logistic Regression': classifier_logistic_regression,
    'MLP': mlp,
    'TabNet': tabnet
}

for model_name, model in models.items():
    acc, prec, rec, f1 = evaluate_model(model, x_test, y_test)
    results['Model'].append(model_name)
    results['Accuracy'].append(acc)
    results['Precision'].append(prec)
    results['Recall'].append(rec)
    results['F1 Score'].append(f1)

# Convert results to DataFrame and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('model_evaluation_results_no_ros.csv', index=False)


# ROS only on training data

In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.neural_network import MLPClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import joblib
#from pytorch_tabnet.tab_model import TabNetClassifier

import os

# Defining the path for the csv file
path = os.path.join("dataset.csv")

# Storing the dataframe in a variable named dataset
dataset = pd.read_csv(path)

# Dropping the unnecessary columns
dataset = dataset.drop('seqn', axis='columns')
dataset = dataset.drop('Marital', axis='columns')
print(dataset.shape)
print(len(dataset[dataset['MetabolicSyndrome'] == 0]))
print(len(dataset[dataset['MetabolicSyndrome'] == 1]))

In [ ]:
# Encoding the categorical variables and filling in the missing values
sex_mapping = {'Male': 0, 'Female': 1}
race_mapping = {'White': 0, 'Asian': 1, 'Black': 2, 'MexAmerican': 3, 'Hispanic': 4, 'Other': 5}

dataset['Sex'] = dataset['Sex'].replace(sex_mapping)
dataset['Race'] = dataset['Race'].replace(race_mapping)

# Fill NaN values in column with index 2, 4, and 5
dataset.iloc[:, 2] = dataset.iloc[:, 2].fillna(dataset.iloc[:, 2].mean())
dataset.iloc[:, 4] = dataset.iloc[:, 4].fillna(dataset.iloc[:, 4].mean())
dataset.iloc[:, 5] = dataset.iloc[:, 5].fillna(dataset.iloc[:, 5].mean())

In [ ]:
# Define the features and target variable
x = dataset.drop('MetabolicSyndrome', axis=1)
y = dataset['MetabolicSyndrome']

ros = RandomOverSampler(random_state=42)
# Perform random over sampling

# Combine the resampled features and target variable into a new dataframe
# Splitting into training and testing data
outcome_0 = dataset[dataset['MetabolicSyndrome'] == 0]
outcome_1 = dataset[dataset['MetabolicSyndrome'] == 1]


test_size_each_class = 474
test_0 = outcome_0.sample(n=test_size_each_class, random_state=42)
test_1 = outcome_1.sample(n=test_size_each_class, random_state=42)
test_data = pd.concat([test_0, test_1])

# Remove the test set rows from the original dataset to create the training set
train_data = dataset.drop(test_data.index)


x_train = train_data.drop('MetabolicSyndrome', axis=1).values
y_train = train_data['MetabolicSyndrome'].values
x_test = test_data.drop('MetabolicSyndrome', axis=1).values
y_test = test_data['MetabolicSyndrome'].values
x_train, y_train = ros.fit_resample(x_train, y_train)

In [ ]:
random_forest_classifier = RandomForestClassifier(n_estimators=30, criterion='entropy', random_state=0)
classifier_decision_tree = DecisionTreeClassifier(criterion='entropy')
xgb_classifer = XGBClassifier(n_estimators = 100, max_depth = 3, learning_rate = 0.5)
classifier_logistic_regression = LogisticRegression()
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=10)
tabnet = TabNetClassifier()

random_forest_classifier.fit(x_train, y_train)
classifier_decision_tree.fit(x_train, y_train)
xgb_classifer.fit(x_train, y_train)
classifier_logistic_regression.fit(x_train, y_train)
mlp.fit(x_train, y_train)
tabnet.fit(
    x_train, y_train,
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

tabnet_pred = tabnet.predict(x_test)
rf_pred = random_forest_classifier.predict(x_test)
dt_pred = classifier_decision_tree.predict(x_test)
mlp_pred = mlp.predict(x_test)
tabnet_pred = tabnet.predict(x_test)
xgb_pred = xgb_classifer.predict(x_test)

# Function to evaluate the model
def evaluate_model(model, x_test, y_test):
    accuracy = []
    precision = []
    recall = []
    f1 = []
    for _ in range(3):
        y_pred = model.predict(x_test)
        accuracy.append(accuracy_score(y_test, y_pred))
        precision.append(precision_score(y_test, y_pred))
        recall.append(recall_score(y_test, y_pred))
        f1.append(f1_score(y_test, y_pred))
    return np.mean(accuracy), np.mean(precision), np.mean(recall), np.mean(f1)

# Evaluate each model
results = {
    'Model': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1 Score': []
}

models = {
    'Random Forest': random_forest_classifier,
    'Decision Tree': classifier_decision_tree,
    'XGBoost': xgb_classifer,
    'Logistic Regression': classifier_logistic_regression,
    'MLP': mlp,
    'TabNet': tabnet
}

for model_name, model in models.items():
    acc, prec, rec, f1 = evaluate_model(model, x_test, y_test)
    results['Model'].append(model_name)
    results['Accuracy'].append(acc)
    results['Precision'].append(prec)
    results['Recall'].append(rec)
    results['F1 Score'].append(f1)

# Convert results to DataFrame and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('model_evaluation_results_ros_training.csv', index=False)
